In [15]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """
    The basic building block of U-Net:
    Conv2d → BatchNorm → ReLU → Conv2d → BatchNorm → ReLU
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)


# Test it
block = DoubleConv(in_channels=1, out_channels=64)
x = torch.zeros(1, 1, 256, 256)
output = block(x)
print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print("Parameters:", sum(p.numel() for p in block.parameters()))

Input shape:  torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 64, 256, 256])
Parameters: 37696


In [16]:
class EncoderBlock(nn.Module):
    """
    One level of the encoder:
    DoubleConv → output saved as skip connection
    MaxPool2d  → downsampled output passed to next level
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        # Apply double conv
        skip = self.conv(x)
        # Downsample
        pooled = self.pool(skip)
        # Return BOTH — skip for the skip connection, pooled to go deeper
        return skip, pooled


# Test it
enc_block = EncoderBlock(in_channels=1, out_channels=64)
x = torch.zeros(1, 1, 256, 256)
skip, pooled = enc_block(x)
print("Input shape:  ", x.shape)
print("Skip shape:   ", skip.shape)
print("Pooled shape: ", pooled.shape)

Input shape:   torch.Size([1, 1, 256, 256])
Skip shape:    torch.Size([1, 64, 256, 256])
Pooled shape:  torch.Size([1, 64, 128, 128])


In [17]:
class DecoderBlock(nn.Module):
    """
    One level of the decoder:
    Upsample → concatenate skip connection → DoubleConv
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, 
                                            kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x, skip):
        # Upsample
        x = self.upsample(x)
        # Concatenate skip connection along channel dimension
        x = torch.cat([skip, x], dim=1)
        # Double conv
        x = self.conv(x)
        return x

# Test it
dec_block = DecoderBlock(in_channels=64, out_channels=32)
# Simulate what would come from the bottleneck
x        = torch.zeros(1, 64, 128, 128)  # upsampled input
skip     = torch.zeros(1, 32, 256, 256)  # skip from encoder



In [18]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Upsample halves the channels: in_channels → in_channels//2
        self.upsample = nn.ConvTranspose2d(in_channels, in_channels//2, 
                                            kernel_size=2, stride=2)
        # After concat: in_channels//2 + in_channels//2 = in_channels
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x, skip):
        x = self.upsample(x)              # in_channels → in_channels//2
        x = torch.cat([skip, x], dim=1)   # in_channels//2 + in_channels//2 = in_channels
        x = self.conv(x)                  # in_channels → out_channels
        return x

# Correct test
dec_block = DecoderBlock(in_channels=128, out_channels=64)
x    = torch.zeros(1, 128, 128, 128)  # from deeper level
skip = torch.zeros(1,  64, 256, 256)  # from encoder — in_channels//2

output = dec_block(x, skip)
print("Input shape: ", x.shape)
print("Skip shape:  ", skip.shape)
print("Output shape:", output.shape)

Input shape:  torch.Size([1, 128, 128, 128])
Skip shape:   torch.Size([1, 64, 256, 256])
Output shape: torch.Size([1, 64, 256, 256])


In [20]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__()
        
        # Encoder
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)
        
        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)
        
        # Decoder
        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)
        
        # Final 1x1 conv → output segmentation map
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)
    
    def forward(self, x):
        # Encoder path
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder path
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)
        
        # Final output
        return self.final_conv(x)


# Test the full U-Net
model = UNet(in_channels=1, num_classes=3)
x = torch.zeros(1, 1, 256, 256)
output = model(x)

print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Input shape:  torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
Total parameters: 31036611
